# Get "missing" nodes file for downstream analysis

"Missing-data bias, particularly in the Y2H network, is not addressed. The authors should examine whether proteins absent from each network differ systematically in abundance, localization, disorder, stability, essentiality, or other relevant characteristics. For example, long ORFs are harder to clone and are underrepresented in the interactome datasets."

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from tqdm import tqdm, trange

## Create "out-of-network" input file

In [7]:
NETWORK_DATA_PATH = Path("../../1-network-centrality/processed-data/")
FASTA_PATH = Path("../../0-download-inputs/data-files/orf_trans.fasta")
OUTPUT_PATH = Path("processed-data/missing-nodes.csv")

# Read network node files.
AEMS_nodes = pd.read_csv(
    NETWORK_DATA_PATH / "0-s288c-step1.csv"
)
Y2H_nodes = pd.read_csv(
    NETWORK_DATA_PATH / "y2h-test-s288c-step1.csv"
)


def extract_network_ids(nodes):
    """Collect all identifiers, including those in grouped nodes."""
    nodes = nodes.copy()
    nodes.columns = nodes.columns.str.strip()

    if "node" not in nodes.columns:
        raise ValueError("Network node file is missing the 'node' column.")

    identifiers = set()

    for value in nodes["node"].dropna().astype(str):
        identifiers.update(
            identifier.strip()
            for identifier in value.split(";")
            if identifier.strip()
        )

    return identifiers


AEMS_ids = extract_network_ids(AEMS_nodes)
Y2H_ids = extract_network_ids(Y2H_nodes)

# Treat the first word of each FASTA header as the authoritative protein ID.
reference_ids = set()
record_count = 0

with FASTA_PATH.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, start=1):
        if line.startswith(">"):
            header = line[1:].strip()

            if not header:
                raise ValueError(
                    f"Empty FASTA header at line {line_number} in {FASTA_PATH}"
                )

            reference_ids.add(header.split()[0])
            record_count += 1

if not reference_ids:
    raise ValueError(f"No FASTA identifiers found in {FASTA_PATH}")

# Keep reference proteins that appear in neither network.
absent_from_both = reference_ids - (AEMS_ids | Y2H_ids)

# Write a single-column node CSV.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"node": sorted(absent_from_both)}).to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"FASTA records: {record_count:,}")
print(f"Unique reference proteins: {len(reference_ids):,}")
print(f"Reference proteins appearing in AE-MS: {len(reference_ids & AEMS_ids):,}")
print(f"Reference proteins appearing in Y2H: {len(reference_ids & Y2H_ids):,}")
print(f"Reference proteins absent from both networks: {len(absent_from_both):,}")
print(f"Output: {OUTPUT_PATH}")

FASTA records: 6,039
Unique reference proteins: 6,039
Reference proteins appearing in AE-MS: 3,928
Reference proteins appearing in Y2H: 1,960
Reference proteins absent from both networks: 1,622
Output: processed-data/missing-nodes.csv


In [6]:
raw_nodes = AEMS_nodes["node"].dropna().astype(str).str.strip()
raw_nodes = raw_nodes.loc[raw_nodes.ne("")]

individual_ids = set(raw_nodes.loc[~raw_nodes.str.contains(";", regex=False)])
grouped_nodes = raw_nodes.loc[raw_nodes.str.contains(";", regex=False)]

group_only_reference_ids = (AEMS_ids - individual_ids) & reference_ids

print(f"AE-MS node rows: {len(AEMS_nodes):,}")
print(f"Unique nonempty node labels: {raw_nodes.nunique():,}")
print(f"Semicolon-delimited node rows: {len(grouped_nodes):,}")
print(f"Unique identifiers after splitting: {len(AEMS_ids):,}")
print(f"Individual identifiers in FASTA: {len(individual_ids & reference_ids):,}")
print(f"Group-only identifiers in FASTA: {len(group_only_reference_ids):,}")

print("\nReference identifiers represented only within grouped nodes:")
for identifier in sorted(group_only_reference_ids):
    matching_groups = grouped_nodes.loc[
        grouped_nodes.apply(
            lambda value: identifier in {
                part.strip() for part in value.split(";")
            }
        )
    ]
    print(f"  {identifier}: {matching_groups.tolist()}")

AE-MS node rows: 3,927
Unique nonempty node labels: 3,927
Semicolon-delimited node rows: 41
Unique identifiers after splitting: 3,939
Individual identifiers in FASTA: 3,876
Group-only identifiers in FASTA: 52

Reference identifiers represented only within grouped nodes:
  YAR009C: ['YER138C;YAR009C']
  YBL100W-A: ['YBL100W-A;YCL020W;YDR034C-C;YFL002W-B;YGR161W-A;YLR410W-A;YLR424C-B;YOR343W-A']
  YCL019W: ['YCL019W;YDR034C-D;YFL002W-A;YGR161W-B']
  YCL020W: ['YBL100W-A;YCL020W;YDR034C-C;YFL002W-B;YGR161W-A;YLR410W-A;YLR424C-B;YOR343W-A']
  YDL083C: ['YMR143W;YDL083C']
  YDR025W: ['YDR025W;YBR048W']
  YDR034C-C: ['YBL100W-A;YCL020W;YDR034C-C;YFL002W-B;YGR161W-A;YLR410W-A;YLR424C-B;YOR343W-A']
  YDR034C-D: ['YCL019W;YDR034C-D;YFL002W-A;YGR161W-B']
  YDR038C: ['YDR039C;YDR038C']
  YDR039C: ['YDR039C;YDR038C']
  YDR225W: ['YDR225W;YBL003C']
  YDR316W-B: ['YDR316W-B;YPR158W-B']
  YDR545W: ['YDR545W;YLR466W;YLR467W;YOR396W;YIL177C;YJL225C']
  YEL077C: ['YER190W;YGR296W;YNL339C;YPL283C;YEL077C